# Module 6: API-Driven Orchestration

**Concepts:** HTTP · `requests` (or stdlib) · GET/POST · query params · auth headers · timeouts

**Audience:** Glue GitLab, Vault, Kubernetes API, metrics — with retries and clear errors.

---

**Note:** Examples use **mocked** HTTP so the notebook runs offline. Install for real calls: `pip install requests`.


## 1. HTTP verbs (mental model)

- **GET** — read (get pipeline status)
- **POST** — create/trigger (start pipeline, login)
- **PUT/PATCH** — update
- **DELETE** — remove

Status codes: 2xx success, 4xx client, 5xx server.


In [ ]:
def classify_status(code: int) -> str:
    if 200 <= code <= 299:
        return "ok"
    if 400 <= code <= 499:
        return "client"
    if 500 <= code <= 599:
        return "server"
    return "other"


for c in [200, 404, 503]:
    print(c, classify_status(c))


## 2. Query parameters and URL building

`?key=value&key2=value2` — use `params` dict with requests.


In [ ]:
from urllib.parse import urlencode

base = "https://gitlab.example/api/v4/projects"
params = {"search": "mlops", "simple": "true"}
url = f"{base}?{urlencode(params)}"
print(url)


## 3. Headers and token auth

Bearer tokens and private tokens go in headers — never commit them; use env vars in real scripts.


In [ ]:
import os

def auth_headers(token_env: str = "GITLAB_TOKEN") -> dict:
    token = os.environ.get(token_env, "REDACTED-demo-token")
    return {"PRIVATE-TOKEN": token}


print(auth_headers())


## 4. Timeouts — always set them

Hung CI glue scripts block deploy trains. Connect + read timeouts.


In [ ]:
DEFAULT_TIMEOUT = (3.0, 10.0)  # connect, read seconds
print("use timeout tuple", DEFAULT_TIMEOUT)


## 5. Mocked `requests` pattern (offline-safe)

Replace network with `unittest.mock` for tests and notebooks.


In [ ]:
from unittest.mock import Mock, patch


def fetch_pipeline_status(project_id: int, pipeline_id: int) -> str:
    import requests

    url = f"https://gitlab.example/api/v4/projects/{project_id}/pipelines/{pipeline_id}"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    return resp.json()["status"]


fake_resp = Mock()
fake_resp.raise_for_status = Mock()
fake_resp.json = Mock(return_value={"status": "success"})

with patch("requests.get", return_value=fake_resp) as get:
    status = fetch_pipeline_status(1, 99)
    print("status", status)
    print("called", get.call_args[0][0])


## 6. POST body (trigger pipeline)

JSON body + headers — same mock pattern for learning.


In [ ]:
from unittest.mock import Mock, patch


def trigger_pipeline(project_id: int, ref: str) -> int:
    import requests

    url = f"https://gitlab.example/api/v4/projects/{project_id}/pipeline"
    resp = requests.post(url, json={"ref": ref}, timeout=10)
    resp.raise_for_status()
    return resp.json()["id"]


fake = Mock()
fake.raise_for_status = Mock()
fake.json = Mock(return_value={"id": 4242})

with patch("requests.post", return_value=fake):
    print("pipeline id", trigger_pipeline(5, "main"))


## 7. Capstone — small client class

Combine base URL, headers, timeout, error handling.


In [ ]:
class GitLabClient:
    def __init__(self, base: str, token: str, timeout: float = 10.0):
        self.base = base.rstrip("/")
        self.headers = {"PRIVATE-TOKEN": token}
        self.timeout = timeout

    def pipeline_url(self, project_id: int, pipeline_id: int) -> str:
        return f"{self.base}/projects/{project_id}/pipelines/{pipeline_id}"


c = GitLabClient("https://gitlab.example/api/v4", "token")
print(c.pipeline_url(1, 2))


---
# Exercises


**Ex 1** — Function building Authorization Bearer header from token string.


In [ ]:
# TODO


**Ex 2** — Mock `requests.get` returning 503; use `raise_for_status` in try/except.


In [ ]:
# TODO


**Ex 3** — Extend `GitLabClient` with method `trigger_url(project_id)`.


In [ ]:
# TODO


---
# Solutions


In [ ]:
def bearer(token: str) -> dict:
    return {"Authorization": f"Bearer {token}"}


print(bearer("abc")["Authorization"][:14])


In [ ]:
from unittest.mock import Mock, patch
import requests


def get_health():
    r = requests.get("https://example/health", timeout=1)
    r.raise_for_status()


bad = Mock()
bad.raise_for_status = Mock(side_effect=requests.HTTPError("503"))

with patch("requests.get", return_value=bad):
    try:
        get_health()
    except requests.HTTPError as e:
        print("handled", e)


In [ ]:
class GitLabClient:
    def __init__(self, base: str, token: str):
        self.base = base.rstrip("/")
        self.headers = {"PRIVATE-TOKEN": token}

    def trigger_url(self, project_id: int) -> str:
        return f"{self.base}/projects/{project_id}/pipeline"


print(GitLabClient("https://g/api/v4", "t").trigger_url(7))


**Next:** Module 7 — types and pytest.
